# Problem Setup
Here we use the equilibrium equations of mechanics to solve the solid mechanics problem

$$ \nabla \cdot \sigma =0 $$

### Nerual Network Structures
Here the output of the neural network uses a multi-output structure, i.e., it predicts both the displacement field and the stress field.

# Libraries

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.autograd import grad
from pyevtk.hl import gridToVTK
from scipy.spatial import KDTree

from Lib.ImportLibrary import configure_notebook
from Lib.IntegrationLoss import IntegrationLoss
from Lib.MultiLayerNet import MultiLayerNet

dev = configure_notebook()
output_dir = Path("output/Case7")
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint = output_dir / "model.pt"

# Training parameters

In [ ]:
steps = 50000
lr = 1e-1
layers = [2,30,30,30,30,6]

## To generate new data:
c = np.array([[66.6,28,0],[28,66.6,0],[0,0,19.3]])

xmin = 0
ymin = 0

Length = 2
Height = 1

known_left_ux = 0
known_left_uy = 0

known_right_tx = 5
known_right_ty = 0

known_up_tx = 0
known_up_ty = 0

known_bottom_tx = 0
known_bottom_ty = 0

# ------------------------------ define domain and collocation points -------------------------------
numElemX = 1000
numElemY = 500
NumLhs = 1000000

x_min, y_min = (0.0, 0.0)
hx = Length / (numElemX - 1)
hy = Height / (numElemY - 1)
shape = [numElemX, numElemY]
dxdy = [hx, hy]

# ------------------------------ data testing -------------------------------------------------------
num_test_x = 1000
num_test_y = 500

# Functions

In [ ]:
def setup_domain():
    x_dom = x_min, Length, numElemX
    y_dom = y_min, Height, numElemY
    # create points
    lin_x = np.linspace(x_dom[0], x_dom[1], x_dom[2])
    lin_y = np.linspace(y_dom[0], y_dom[1], y_dom[2])
    dom = np.zeros((numElemX * numElemY, 2))
    c = 0
    for x in np.nditer(lin_x):
        tb = y_dom[2] * c
        te = tb + y_dom[2]
        c += 1
        dom[tb:te, 0] = x
        dom[tb:te, 1] = lin_y
    print(dom.shape)
    np.meshgrid(lin_x, lin_y)

    # ------------------------------------ BOUNDARY ----------------------------------------
    # Left boundary condition (Dirichlet BC)
    bcl_pts_idx = np.where(dom[:, 0] == x_min)
    bcl_pts = dom[bcl_pts_idx, :][0]
    bcl = np.ones(np.shape(bcl_pts)) * [known_left_ux, known_left_uy]
    bcl_outnormal = np.array([1, 0])
    #

    # Right boundary condition (Neumann BC)
    bcr_pts_idx = np.where(dom[:, 0] == Length - x_min)
    bcr_pts = dom[bcr_pts_idx, :][0]
    bcr = np.ones(np.shape(bcr_pts)) * [known_right_tx, known_right_ty]
    bcr_outnormal = np.array([1, 0])

    # up boundary condition (Neumann BC)
    bcu_pts_idx = np.where(dom[:, 1] == Height - y_min)
    bcu_pts = dom[bcu_pts_idx, :][0]
    bcu_t = np.ones(np.shape(bcu_pts)) * [known_up_tx, known_up_ty]
    bcu_outnormal = np.array([0, 1])

    # Bottom boundary condition (Neumann BC)
    bcb_pts_idx = np.where(dom[:, 1] == y_min)
    bcb_pts = dom[bcb_pts_idx, :][0]
    bcb_t = np.ones(np.shape(bcb_pts)) * [known_bottom_tx, known_bottom_ty]
    bcb_outnormal = np.array([0, 1])

    boundary_neumann = {
        # condition on the right
        "RIGHT": {
            "coord": bcr_pts,
            "known_value": bcr,
            "out_normal": bcr_outnormal
        },
        "UP": {
            "coord": bcu_pts,
            "known_value": bcu_t,
            "out_normal": bcu_outnormal
        },
        "BOTTOM": {
            "coord": bcb_pts,
            "known_value": bcb_t,
            "out_normal": bcb_outnormal
        }
    }
    boundary_dirichlet = {
        # condition on the left
        "dirichlet_1": {
            "coord": bcl_pts,
            "known_value": bcl,
            "out_normal": bcl_outnormal
        }
        # adding more boundary condition here ...
    }
    return dom, boundary_neumann, boundary_dirichlet

def get_datatest():
    x_dom_test = x_min, Length, num_test_x
    y_dom_test = y_min, Height, num_test_y
    # create points
    x_space = np.linspace(x_dom_test[0], x_dom_test[1], x_dom_test[2])
    y_space = np.linspace(y_dom_test[0], y_dom_test[1], y_dom_test[2])
    xGrid, yGrid = np.meshgrid(x_space, y_space)
    data_test = np.concatenate(
        (np.array([xGrid.flatten()]).T, np.array([yGrid.flatten()]).T), axis=1)
    return x_space, y_space, data_test

def writer_vtk(datatest, fileout):
    x1D = datatest[:, 0].flatten()
    y1D = datatest[:, 1].flatten()

    z = np.array([0]).astype("float64")
    S11, S22, S12, S21, Ux, Uy = PDNN.evaluate_model(x1D, y1D, z)

    S11 = S11.detach().cpu().numpy()
    S12 = S12.detach().cpu().numpy()
    S21 = S21.detach().cpu().numpy()
    S22 = S22.detach().cpu().numpy()
    Ux = Ux.detach().cpu().numpy()
    Uy = Uy.detach().cpu().numpy()
    Nx = len(x)
    Ny = len(y)
    surS11 = S11.reshape(Ny, Nx, 1)
    surS12 = S12.reshape(Ny, Nx, 1)
    surS21 = S21.reshape(Ny, Nx, 1)
    surS22 = S22.reshape(Ny, Nx, 1)
    surUx = Ux.reshape(Ny, Nx, 1)
    surUy = Uy.reshape(Ny, Nx, 1)
    surUz = np.zeros([Nx, Ny, 1])
    xx, yy, zz = np.meshgrid(x, y, z)
    gridToVTK(fileout, xx, yy, zz,
              pointData={"S11": np.float64(surS11), "S12": np.float64(surS12), "S21": np.float64(surS21),
                         "S22": np.float64(surS22), "Ux": np.float64(surUx), "Uy": np.float64(surUy)})

In [ ]:
class PhysicsDrivenNeuralNetwork:
    # Instance attributes
    def __init__(self, model, numIntType, dim):
        self.model = MultiLayerNet(model, init=True)
        self.model = self.model.to(dev)
        self.intLoss = IntegrationLoss(numIntType, dim)
        self.dim = dim
        self.beta = 0.5
        
    def train_model(self, data, marker, dxdy, neumannBC, dirichletBC, iteration, learning_rate):
        marker = torch.as_tensor(marker, device=dev)
        self.lam = torch.ones([len(data), 1]).to(dev)
        data = torch.from_numpy(data).float()
        data = data.to(dev)
        data.requires_grad_(True)
        # ----------------------------------------------------------------------------------
        # Minimizing loss function (energy and boundary conditions)
        # ----------------------------------------------------------------------------------
        optimizer = torch.optim.LBFGS(self.model.parameters(), lr=learning_rate, max_iter=1)
        start_time = time.time()
        it_time = time.time()

        for t in range(iteration):
            # Zero gradients, perform a backward pass, and update the weights.
            def closure():
                # ----------------------------------------------------------------------------------
                # Internal Energy
                # ----------------------------------------------------------------------------------
                u_pred, stress_pred = self.get_pred(data)
                # u_pred.double()

                stress = self.getStress(u_pred, data)

                PDE_x, PDE_y = self.getPDELoss(u_pred, data, stress_pred)

                self.loss_MO = torch.mean(torch.square(stress_pred[:, 0] - stress[:, 0])) \
                    + torch.mean(torch.square(stress_pred[:, 1] - stress[:, 1])) \
                    + torch.mean(torch.square(stress_pred[:, 2] - stress[:, 2])) \
                    + torch.mean(torch.square(stress_pred[:, 3] - stress[:, 3]))
                
                self.PDE_Const_loss = torch.mean(torch.square(PDE_x)) + torch.mean(torch.square(PDE_y))

                self.boundary_loss = self.Boundary_Loss(neumannBC, dirichletBC)
                
                loss_PDE = 10 * (torch.abs(stress[:, 0] - stress_pred[:, 0])
                                + torch.abs(stress[:, 1] - stress_pred[:, 1])
                                + torch.abs(stress[:, 2] - stress_pred[:, 2])
                                + torch.abs(stress[:, 3] - stress_pred[:, 3])) \
                                + torch.abs(PDE_x) \
                                 + torch.abs(PDE_y)

                loss_PDE = loss_PDE.unsqueeze(1)
                residual = loss_PDE.detach().abs()
                total = residual.sum().clamp_min(torch.finfo(residual.dtype).tiny)
                weight_interior = (residual / total) * residual.size(0)
                interior_KNN = weight_interior[marker].squeeze(-1)
                tmp_lam = torch.sum(interior_KNN, 1).reshape(-1, 1) / np.size(marker, 1)
                self.lam = self.beta * tmp_lam + (1 - self.beta) * self.lam

                # loss = self.PDE_Const_loss + self.loss_MO * 10 + self.boundary_loss * 10
                
                loss = torch.mean(torch.square(loss_PDE * self.lam)) + self.boundary_loss

                optimizer.zero_grad()
                loss.backward()
                return loss
            optimizer.step(closure)

            if t % 100 == 0:
                print('Iter: %d Loss: %.5e loss_PDE: %.5e loss_MO: %.5e Boundary: %.5e Time: %.3e'
                        % (t + 1, closure().item(), self.PDE_Const_loss.item(), self.loss_MO.item(),self. boundary_loss.item(), time.time() - it_time))
                torch.save(self.model.state_dict(), checkpoint)
                it_time = time.time()



    def get_pred(self, x):
        pred = self.model(x)
        u = pred[:, 0:2]
        P_pred = pred[:, 2:6]
        Ux = x[:, 0] * u[:, 0]
        Uy = x[:, 0] * u[:, 1]
        Ux = Ux.reshape(Ux.shape[0], 1)
        Uy = Uy.reshape(Uy.shape[0], 1)
        u_pred = torch.cat((Ux, Uy), -1)
        return u_pred, P_pred
    
    def Boundary_Loss(self, neumannBC, dirichletBC):
        # get tensor inputs and outputs for boundary conditions
        # -------------------------------------------------------------------------------
        #                             Dirichlet BC
        # -------------------------------------------------------------------------------
        dirBC_coordinates = {}  # declare a dictionary
        dirBC_values = {}  # declare a dictionary
        dirBC_outnormal = {}  # declare a dictionary
        for i, keyi in enumerate(dirichletBC):
            dirBC_coordinates[i] = torch.from_numpy(dirichletBC[keyi]['coord']).float().to(dev)
            dirBC_coordinates[i].requires_grad_(True)
            dirBC_values[i] = torch.from_numpy(dirichletBC[keyi]['known_value']).float().to(dev)
            dirBC_outnormal[i] = torch.from_numpy(dirichletBC[keyi]['out_normal']).float().to(dev)
        # -------------------------------------------------------------------------------
        #                           Neumann BC
        # -------------------------------------------------------------------------------
        neuBC_coordinates = {}  # declare a dictionary
        neuBC_values = {}  # declare a dictionary
        neuBC_outnormal = {}  # declare a dictionary
        for i, keyi in enumerate(neumannBC):
            neuBC_coordinates[i] = torch.from_numpy(neumannBC[keyi]['coord']).float().to(dev)
            neuBC_coordinates[i].requires_grad_(True)
            neuBC_values[i] = torch.from_numpy(neumannBC[keyi]['known_value']).float().to(dev)
            neuBC_outnormal[i] = torch.from_numpy(neumannBC[keyi]['out_normal']).float().to(dev)
        # ----------------------------------------------------------------------------------
        # Neumann BC
        # ----------------------------------------------------------------------------------
        # Right
        _, S_right = self.get_pred(neuBC_coordinates[0])
        outnormal = neuBC_outnormal[0]
        S_x = S_right[:, 0] * outnormal[0] + S_right[:, 2] * outnormal[1]
        S_y = S_right[:, 3] * outnormal[0] + S_right[:, 1] * outnormal[1]
        loss_right = torch.mean(torch.square(S_x - neuBC_values[0][:, 0])) \
                     + torch.mean(torch.square(S_y - neuBC_values[0][:, 1]))

        # Up
        _, S_up = self.get_pred(neuBC_coordinates[1])
        outnormal = neuBC_outnormal[1]
        S_x = S_up[:, 0] * outnormal[0] + S_up[:, 2] * outnormal[1]
        S_y = S_up[:, 3] * outnormal[0] + S_up[:, 1] * outnormal[1]
        loss_up = torch.mean(torch.square(S_x - neuBC_values[1][:, 0])) \
                  + torch.mean(torch.square(S_y - neuBC_values[1][:, 1]))

        # Bottom
        _, S_bottom = self.get_pred(neuBC_coordinates[2])
        outnormal = neuBC_outnormal[2]
        S_x = S_bottom[:, 0] * outnormal[0] + S_bottom[:, 2] * outnormal[1]
        S_y = S_bottom[:, 3] * outnormal[0] + S_bottom[:, 1] * outnormal[1]
        loss_bottom = torch.mean(torch.square(S_x - neuBC_values[2][:, 0])) \
                      + torch.mean(torch.square(S_y - neuBC_values[2][:, 1]))

        # ----------------------------------------------------------------------------------
        # Dirichlet BC
        # ----------------------------------------------------------------------------------
        # Left
        u_left, _ = self.get_pred(dirBC_coordinates[0])
        loss_left = torch.mean(torch.square(u_left[:, 0] - dirBC_values[0][:, 0])) \
                    + torch.mean(torch.square(u_left[:, 1] - dirBC_values[0][:, 1]))

        loss_bc = loss_up + loss_left + loss_bottom + loss_right

        return loss_bc

    def getPDELoss(self, u, x, S):
        S11 = S[:, 0]
        S22 = S[:, 1]
        S12 = S[:, 2]
        S21 = S[:, 3]

        dS11dxy = grad(S11.unsqueeze(1), x, torch.ones(x.size()[0], 1, device=dev),
                       create_graph=True, retain_graph=True)[0]
        dS12dxy = grad(S12.unsqueeze(1), x, torch.ones(x.size()[0], 1, device=dev),
                       create_graph=True, retain_graph=True)[0]
        dS21dxy = grad(S21.unsqueeze(1), x, torch.ones(x.size()[0], 1, device=dev),
                       create_graph=True, retain_graph=True)[0]
        dS22dxy = grad(S22.unsqueeze(1), x, torch.ones(x.size()[0], 1, device=dev),
                       create_graph=True, retain_graph=True)[0]

        x_axis = dS11dxy[:, 0] + dS12dxy[:, 1]
        y_axis = dS21dxy[:, 0] + dS22dxy[:, 1]

        return x_axis, y_axis
    
    def getStress(self, u, x):
        duxdxy = grad(u[:, 0].unsqueeze(1), x, torch.ones(x.size()[0], 1, device=dev),
                    create_graph=True, retain_graph=True)[0]
        duydxy = grad(u[:, 1].unsqueeze(1), x, torch.ones(x.size()[0], 1, device=dev),
                      create_graph=True, retain_graph=True)[0]

        e11 = duxdxy[:, 0].unsqueeze(1)
        e22 = duydxy[:, 1].unsqueeze(1)
        e12 = 0.5 * (duxdxy[:, 1].unsqueeze(1) + duydxy[:, 0].unsqueeze(1))

        Sigma11 = c[0,0] * e11 + c[0,1] * e22 + c[0,2] * e12
        Sigma22 = c[1,0] * e11 + c[1,1] * e22 + c[1,2] * e12
        Sigma12 = c[2,0] * e11 + c[2,1] * e22 + c[2,2] * e12
        Sigma21 = c[2,0] * e11 + c[2,1] * e22 + c[2,2] * e12

        Sigma = torch.cat((Sigma11, Sigma22, Sigma12, Sigma21), -1)
        return Sigma
    
    def evaluate_model(self, x, y, z, Typle=None, OP=False):
        xy = np.concatenate((np.array([x]).T, np.array([y]).T), axis=-1)
        xy_tensor = torch.from_numpy(xy).float()
        xy_tensor = xy_tensor.to(dev)
        xy_tensor.requires_grad_(True)

        u_pred_torch, _ = self.get_pred(xy_tensor)

        S = self.getStress(u_pred_torch, xy_tensor)

        S11 = S[:, 0].unsqueeze(1)
        S22 = S[:, 1].unsqueeze(1)
        S12 = S[:, 2].unsqueeze(1)
        S21 = S[:, 3].unsqueeze(1)

        Ux = u_pred_torch[:, 0]
        Uy = u_pred_torch[:, 1]
        return S11, S22, S12, S21, Ux, Uy

In [ ]:
dom, boundary_neumann, boundary_dirichlet = setup_domain()
x, y, datatest = get_datatest()

tree = KDTree(dom)
[_, marker] = tree.query(dom, 10)

In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)
ax.scatter(dom[:, 0], dom[:, 1],marker='o', s=5,vmin=0, vmax=1)
ax.set_xlabel('X', fontsize=3)
ax.set_ylabel('Y', fontsize=3)
ax.tick_params(labelsize=4)
plt.axis('equal')
plt.show()

In [ ]:
PDNN = PhysicsDrivenNeuralNetwork(layers, 'simpson', 2)

In [ ]:
start_time = time.time()
PDNN.train_model(dom, marker, dxdy, boundary_neumann, boundary_dirichlet, steps, lr)
end_time = time.time() - start_time
print("End time: %.5f" % end_time)
torch.save(PDNN.model.state_dict(), checkpoint)

In [ ]:
PDNN.model.load_state_dict(torch.load(checkpoint, map_location=dev, weights_only=True))
PDNN.model.eval()

writer_vtk(datatest, str(output_dir / "output"))